# Visualize Exported Batch Summary

Use the exported PKL/CSV table to inspect parameter-observation relations, observation pairplots, and PCA coverage.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from export_summary_visualization import (
    load_export_summary_table,
    find_priors_batch_pickle,
    load_priors_batch_metadata,
    infer_parameter_sampling_modes,
    plot_parameter_observation_grid,
    plot_observation_pairgrid,
    plot_observation_pca,
)


In [ ]:
# PATHS
BATCH_ROOT_DIR = Path("sim_batches_BURST_properties_v2")
# EXPORT_TABLE_PATH = BATCH_ROOT_DIR / "parameter_set_summaries" / "parameter_set_feature_summary.pkl"
EXPORT_TABLE_PATH = BATCH_ROOT_DIR / "parameter_set_summaries_FULL" / "parameter_set_feature_summary.pkl"
PRIORS_BATCH_PATH = None  # None -> auto-detect *_priors_batch*.pkl in BATCH_ROOT_DIR
# OUTPUT_DIR = BATCH_ROOT_DIR / "export_summary_visualizations"
OUTPUT_DIR = BATCH_ROOT_DIR / "parameter_set_summaries_FULL"

# FIGURE EXPORT
SAVE_FIGURES = True

# SELECT VARIABLES
# List-valued parameters such as *_band_hz are usually better left out of scatter grids
# unless you intentionally want them treated as categorical axes.
PARAMETER_COLUMNS = [
    # "SIM_PARAM_GENERAL_duration",
    # "SIM_PARAM_GENERAL_nb_motoneurons",
    # "SIM_PARAM_GENERAL_min_soma_diameter",
    # "SIM_PARAM_GENERAL_max_soma_diameter",
    # "SIM_PARAM_GENERAL_tau_size_mode",
    # "SIM_PARAM_GENERAL_tau_size_ratio",
    # "SIM_PARAM_GENERAL_tau_size_um",
    # "SIM_PARAM_BASELINE_INPUT_excitatory_input_baseline",
    "SIM_PARAM_BASELINE_INPUT_lf_target_sd",
    "SIM_PARAM_BASELINE_INPUT_alpha_target_sd_final",
    "SIM_PARAM_BASELINE_INPUT_beta_target_sd_final",
    # "SIM_PARAM_BASELINE_INPUT_independent_input_power",
    #
    "SIM_PARAM_BURST_INPUT_input_burst_sigma_ms",
    "SIM_PARAM_BURST_INPUT_lf_burst_peak_nA",
    "SIM_PARAM_BURST_INPUT_alpha_burst_peak_nA",
    "SIM_PARAM_BURST_INPUT_beta_burst_peak_nA",
]

OBSERVATION_COLUMNS = [
    "OBS_FIRING_STATISTICS_firing_rate_mean",
    "OBS_FIRING_STATISTICS_firing_rate_sd",
    "OBS_FIRING_STATISTICS_firing_rate_skew",
    "OBS_FIRING_STATISTICS_isi_cv_mean",
    "OBS_FIRING_STATISTICS_isi_cv_sd",
    "OBS_FIRING_STATISTICS_isi_cv_skew",
    "OBS_BASELINE_cst_low_pass_mean_spikes_per_s_per_MU",
    "OBS_BASELINE_cst_alpha_hilbert_mean_spikes_per_s_per_MU",
    "OBS_BASELINE_cst_beta_hilbert_mean_spikes_per_s_per_MU",
    "OBS_BURST_FEATURES_resid_cst_low_frequency",
    "OBS_BURST_FEATURES_resid_cst_alpha_modulation_hilbert_zscore",
    "OBS_BURST_FEATURES_resid_cst_beta_modulation_hilbert_zscore",
    "OBS_BURST_FEATURES_resid_sync_coincidence_curve",
]

MULTILINE_PREFIX_LABELS = True # obs and params prefixes and names are displayed on different lines
SHARE_AXIS_LIMITS_BY_UNIT = True
DISPLAY_CURRENT_UNIT = "µA"  # "µA" or "nA" for current-like variables saved in nA

# Unit groups used for label display and optional shared axes.
UNIT_GROUPS = {
    "µm": [
        "SIM_PARAM_GENERAL_min_soma_diameter",
        "SIM_PARAM_GENERAL_max_soma_diameter",
        "SIM_PARAM_GENERAL_tau_size_um",
    ],
    "ms": [
        "SIM_PARAM_BURST_INPUT_input_burst_sigma_ms",
    ],
    "nA": [
        "SIM_PARAM_BASELINE_INPUT_excitatory_input_baseline",
        "SIM_PARAM_BASELINE_INPUT_lf_target_sd",
        "SIM_PARAM_BASELINE_INPUT_alpha_target_sd_final",
        "SIM_PARAM_BASELINE_INPUT_beta_target_sd_final",
        "SIM_PARAM_BASELINE_INPUT_independent_input_power",
        "SIM_PARAM_BURST_INPUT_lf_burst_peak_nA",
        "SIM_PARAM_BURST_INPUT_alpha_burst_peak_nA",
        "SIM_PARAM_BURST_INPUT_beta_burst_peak_nA",
    ],
    "spikes/s/MU (fluctuations)": [
        "OBS_BASELINE_cst_alpha_hilbert_mean_spikes_per_s_per_MU",
        "OBS_BASELINE_cst_beta_hilbert_mean_spikes_per_s_per_MU",
    ],
    "spikes/s/MU (mean)": [
        "OBS_FIRING_STATISTICS_firing_rate_mean",
        "OBS_BASELINE_cst_low_pass_mean_spikes_per_s_per_MU",
    ],
    "spikes/s/MU (sd)": [   
        "OBS_FIRING_STATISTICS_firing_rate_sd",
    ],
    "ISI CV (mean)": [
        "OBS_FIRING_STATISTICS_isi_cv_mean",
    ],
    "ISI CV (sd)": [
        "OBS_FIRING_STATISTICS_isi_cv_sd",
    ],
    "z": [
        "OBS_BURST_FEATURES_resid_cst_alpha_modulation_hilbert_zscore",
        "OBS_BURST_FEATURES_resid_cst_beta_modulation_hilbert_zscore",
    ],
    "a.u. sync": [
        "OBS_BURST_FEATURES_resid_sync_coincidence_curve",
    ],
    "a.u. skew": [
        "OBS_FIRING_STATISTICS_firing_rate_skew",
        "OBS_FIRING_STATISTICS_isi_cv_skew",
    ],
    "s": [
        "SIM_PARAM_GENERAL_duration",
        "SIM_PARAM_GENERAL_get_ready_cue_time_s",
        "SIM_PARAM_GENERAL_go_nogo_cue_delay_s",
        "SIM_PARAM_BURST_INPUT_burst_delay_after_go_nogo_cue_s",
    ],
}

# Observation colors. These are used directly in the parameter-vs-observation grid,
# and in the observation pairplot (diagonal + row color).
OBSERVATION_COLORS = {
    "OBS_FIRING_STATISTICS_firing_rate_mean": "#E67E22",
    "OBS_FIRING_STATISTICS_firing_rate_sd": "#E67E22",
    "OBS_FIRING_STATISTICS_firing_rate_skew": "#E67E22",
    "OBS_FIRING_STATISTICS_isi_cv_mean": "#00B894",
    "OBS_FIRING_STATISTICS_isi_cv_sd": "#00B894",
    "OBS_FIRING_STATISTICS_isi_cv_skew": "#00B894",
    "OBS_BASELINE_cst_low_pass_mean_spikes_per_s_per_MU": "#ff3300",
    "OBS_BASELINE_cst_alpha_hilbert_mean_spikes_per_s_per_MU": "#ff9900",
    "OBS_BASELINE_cst_beta_hilbert_mean_spikes_per_s_per_MU": "#008cff",
    "OBS_BURST_FEATURES_resid_cst_low_frequency": "#ff3300",
    "OBS_BURST_FEATURES_resid_cst_alpha_modulation_hilbert_zscore": "#ff9900",
    "OBS_BURST_FEATURES_resid_cst_beta_modulation_hilbert_zscore": "#008cff",
    "OBS_BURST_FEATURES_resid_sync_coincidence_curve": "#852680",
}

# STYLE
GRID_STYLE = {
    "cell_size": (4.0, 4.0),
    "scatter_color": "#d859e4",
    "scatter_alpha": 0.30,
    "scatter_s": 50,
    "font_family": "DejaVu Sans",
    "label_fontsize": 8,
    "annotation_fontsize": 9,
    "title_fontsize": 10,
    "tick_labelsize": 8,
    "xtick_rotation": 45,
    "show_regression_line": True,
    "regression_lw": 3.0,
    "regression_alpha": 0.50,
    "regression_ls": "-",
}

PAIRPLOT_STYLE = {
    "cell_size": (4.0, 4.0),
    "scatter_color": "#1f77b4",
    "hist_color": "#1f77b4",
    "scatter_alpha": 0.30,
    "scatter_s": 50,
    "font_family": "DejaVu Sans",
    "label_fontsize": 9,
    "title_fontsize": 12,
    "tick_labelsize": 8,
    "kde_cmap": "Blues",
}
PAIRPLOT_SHOW_KDE = False
PAIRPLOT_KDE_ONLY = False

PCA_STYLE = {
    "figure_size": (7.0, 7.0),
    "scatter_color": "#1f77b4",
    "scatter_alpha": 0.30,
    "scatter_s": 50,
    "font_family": "DejaVu Sans",
    "label_fontsize": 8,
    "title_fontsize": 12,
    "tick_labelsize": 9,
    "kde_cmap": "Blues",
}
PCA_SHOW_KDE = False
PCA_KDE_ONLY = False


In [ ]:
export_df = load_export_summary_table(EXPORT_TABLE_PATH)
priors_path = Path(PRIORS_BATCH_PATH) if PRIORS_BATCH_PATH is not None else find_priors_batch_pickle(BATCH_ROOT_DIR)
priors_payload = load_priors_batch_metadata(priors_path)
sampling_modes = infer_parameter_sampling_modes(export_df, priors_payload)
print(f"Loaded export table: {EXPORT_TABLE_PATH}")
print(f"Loaded priors metadata: {priors_path}")
print(f"Rows: {len(export_df)}")
export_df.head()


In [ ]:
param_obs_output_path = OUTPUT_DIR / "parameter_vs_observation_grid.png" if SAVE_FIGURES else None
fig_param_obs = plot_parameter_observation_grid(
    export_df=export_df,
    parameter_columns=PARAMETER_COLUMNS,
    observation_columns=OBSERVATION_COLUMNS,
    sampling_modes=sampling_modes,
    observation_colors=OBSERVATION_COLORS,
    unit_groups=UNIT_GROUPS,
    display_current_unit=DISPLAY_CURRENT_UNIT,
    share_axis_limits_by_unit=SHARE_AXIS_LIMITS_BY_UNIT,
    multiline_labels=MULTILINE_PREFIX_LABELS,
    style=GRID_STYLE,
    output_path=param_obs_output_path,
)
if SAVE_FIGURES:
    print(param_obs_output_path)
plt.show()


In [ ]:
pairplot_output_path = OUTPUT_DIR / "observation_pairplot.png" if SAVE_FIGURES else None
fig_pairplot = plot_observation_pairgrid(
    export_df=export_df,
    observation_columns=OBSERVATION_COLUMNS,
    style=PAIRPLOT_STYLE,
    observation_colors=OBSERVATION_COLORS,
    unit_groups=UNIT_GROUPS,
    display_current_unit=DISPLAY_CURRENT_UNIT,
    multiline_labels=MULTILINE_PREFIX_LABELS,
    show_kde=PAIRPLOT_SHOW_KDE,
    kde_only=PAIRPLOT_KDE_ONLY,
    output_path=pairplot_output_path,
)
if SAVE_FIGURES:
    print(pairplot_output_path)
plt.show()


In [ ]:
pca_output_path = OUTPUT_DIR / "observation_pca.png" if SAVE_FIGURES else None
fig_pca, pca_result = plot_observation_pca(
    export_df=export_df,
    observation_columns=OBSERVATION_COLUMNS,
    style=PCA_STYLE,
    show_kde=PCA_SHOW_KDE,
    kde_only=PCA_KDE_ONLY,
    output_path=pca_output_path,
)
if SAVE_FIGURES:
    print(pca_output_path)
plt.show()
pca_result
